In [ ]:
import torch
import torch.nn.functional as F
import matplotlib
%matplotlib inline

In [ ]:
hiden_size = 100
block_size = 3 # contect window size: how many characters do we take to predict the next one
embedding_size = 2

In [ ]:
with open('names.txt', 'r') as f:
    words = f.read().splitlines()

In [ ]:
len(words)

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s,i in stoi.items()}
print(itos)

In [ ]:
X, Y = [], []


for w in words[:5]:
    print(w)
    context  = [0] * block_size
    for ch in w + '.':
        id = stoi[ch]
        X.append(context)
        Y.append(id)
        print(''.join([itos[c] for c in context]), f' ---> {itos[id]}')
        context = context[1:] + [id]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
X.shape, Y.shape, X.dtype, Y.dtype

In [ ]:
C = torch.randn((27, embedding_size)) # character embedding table

In [ ]:
C[5]

In [ ]:
F.one_hot(torch.tensor(5), num_classes=27).float() @  C

In [ ]:
emb  = C[X]
emb.shape

In [ ]:
W1 = torch.randn((embedding_size * block_size, hiden_size)) 
b1 = torch.randn(hiden_size)
W2 = torch.randn((hiden_size , 27))
b2 = torch.randn(27)

In [ ]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [ ]:
logits = h @ W2 + b2
counts = logits.exp()

In [ ]:
prob = counts/counts.sum(1, keepdims = True)

In [ ]:
prob.shape

In [ ]:
prob[0].sum()

In [ ]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

### Rewriting

In [ ]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, embedding_size), generator = g) # character embedding table
W1 = torch.randn((embedding_size * block_size, hiden_size), generator = g )
b1 = torch.randn(hiden_size, generator = g)
W2 = torch.randn((hiden_size , 27), generator = g)
b2 = torch.randn(27, generator = g)
parameters = [C, W1, W2, b1, b2]

In [ ]:
sum(p.nelement()for p in parameters) 

In [ ]:
emb  = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
counts = logits.exp()
prob = counts/counts.sum(1, keepdims = True)
loss = -prob[torch.arange(32), Y].log().mean()
loss